# Tokenizer impact on genetic variant extraction

This notebook starts from scratch and keeps only the analysis needed to answer:

> Are some variant mentions missed because the tokenizer splits them badly?

It does **not** run LLM extraction again. It only loads your existing evaluation files, inspects tokenization, and relates tokenization features to TP/FN/FP/TN behavior.

## 1. Imports and paths

In [1]:
# If needed, uncomment:
# !pip install pandas numpy matplotlib python-dotenv transformers scikit-learn tqdm

import os
import re
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from transformers import AutoTokenizer
from sklearn.metrics import confusion_matrix, f1_score

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 100)

load_dotenv()
print("Imports loaded.")

Imports loaded.


In [2]:
# -----------------------------
# CONFIGURATION
# -----------------------------
# These defaults mirror your previous notebook.
# Replace them with explicit paths if your .env is not available.

INPUT_DIRECTORY = os.getenv("INPUT_DIRECTORY", ".")
OUTPUT_DIRECTORY = os.getenv("OUTPUT_DIRECTORY", ".")
LLM_EVALUATION_DIRECTORY = os.getenv("LLM_EVALUATION_DIRECTORY", OUTPUT_DIRECTORY)

CANCER_DATA_FILE = "binary_cancer_matrix_filtered.csv"
EVALUATION_FILE = "LLM_evaluation_statistics.csv"

input_path = Path(INPUT_DIRECTORY)
output_path = Path(OUTPUT_DIRECTORY)
evaluation_path = Path(LLM_EVALUATION_DIRECTORY)

print("INPUT_DIRECTORY:", input_path.resolve())
print("OUTPUT_DIRECTORY:", output_path.resolve())
print("LLM_EVALUATION_DIRECTORY:", evaluation_path.resolve())

INPUT_DIRECTORY: /Users/andreiblindu/Desktop/Idiap/code/Variantscape/results_openalex/04_categorization
OUTPUT_DIRECTORY: /Users/andreiblindu/Desktop/Idiap/code/Variantscape/results_openalex/05_LLM_variant_extraction
LLM_EVALUATION_DIRECTORY: /Users/andreiblindu/Desktop/Idiap/code/Variantscape/results_openalex/05_LLM_evaluation


## 2. Load the article dataset and LLM evaluation file

In [3]:
# Load article data: PaperId, PaperTitle, Abstract
cancer_file_path = input_path / CANCER_DATA_FILE

if not cancer_file_path.exists():
    raise FileNotFoundError(
        f"Could not find {cancer_file_path}. "
        "Update INPUT_DIRECTORY or CANCER_DATA_FILE."
    )

cancer_df = pd.read_csv(cancer_file_path)

required_article_cols = ["PaperId", "PaperTitle", "Abstract"]
missing = [c for c in required_article_cols if c not in cancer_df.columns]
if missing:
    raise ValueError(f"Missing required article columns: {missing}")

cancer_df = cancer_df[required_article_cols].copy()

print(f"Loaded article dataset: {len(cancer_df):,} rows")
display(cancer_df.head())

Loaded article dataset: 14,500 rows


,PaperId,PaperTitle,Abstract
0,7106250716,PVNDMVLiver Circuit Drives Hepatocellular Carcinoma Progression in the Context of Depression Comorbidity,"Hepatocellular carcinoma (HCC) and depression frequently co-occur, yet whether and how depression affects HCC progression is unknown. Here, we show that stress-induced mouse depression models, chr..."
1,7119160011,Screening and biological function analysis of differentially expressed genes associated with miR-21 in hepatocellular carcinoma tissue,"Objective To investigate the differentially expressed genes (DEGs) associated with miR-21 between hepatocellular carcinoma (HCC) tissue and normal tissue, as well as their biological function in t..."
2,7125102904,"Recent advances in non-alcoholic steatohepatitis-associated hepatocellular carcinoma: immune cells, metabolic dysregulation, and therapeutic strategies","Non-alcoholic steatohepatitis (NASH), the inflammatory progression of non-alcoholic fatty liver disease (NAFLD), is a leading cause of hepatocellular carcinoma (HCC) amid rising obesity and metabo..."
3,7125105582,"Complete response to BRICS in Locally advanced pancreatic cancer (pMMR, CPS 30): a case report","Background Locally advanced pancreatic cancer (LAPC) has a dismal prognosis, marked by an exceedingly low 5-year survival rate. While immune checkpoint inhibitors (ICIs) have demonstrated efficacy..."
4,7125132542,"Diagnostic value of interleukin-8 in colon cancer: Prospective, case-control study","BACKGROUND Interleukin-8 (IL-8), a pro-inflammatory chemokine, is implicated in angiogenesis, tumor growth, and metastasis. However, its diagnostic and prognostic significance in colorectal cancer..."


In [4]:
# Load LLM evaluation statistics
evaluation_file_path = input_path / EVALUATION_FILE

if not evaluation_file_path.exists():
    # Fallback: sometimes evaluation files are saved in LLM_EVALUATION_DIRECTORY
    evaluation_file_path = evaluation_path / EVALUATION_FILE

if not evaluation_file_path.exists():
    raise FileNotFoundError(
        f"Could not find {EVALUATION_FILE} in {input_path} or {evaluation_path}."
    )

llm_df = pd.read_csv(evaluation_file_path)

if "Human" not in llm_df.columns:
    raise ValueError("Expected a 'Human' column in the evaluation file.")

if "PaperId" not in llm_df.columns:
    print("Warning: no PaperId column found in the evaluation file. Some article-level analyses will be limited.")

# Automatically infer model columns as all non-metadata columns except Human.
metadata_like = {
    "PaperId", "PaperTitle", "Abstract", "Title", "PMID", "DOI",
    "LLM_Prompt", "Prompt", "Notes"
}
model_columns = [
    c for c in llm_df.columns
    if c not in metadata_like and c != "Human"
]

print(f"Loaded evaluation file: {len(llm_df):,} rows")
print("Detected model columns:")
print(model_columns)

display(llm_df.head())

Loaded evaluation file: 797 rows
Detected model columns:
['Lang', 'PubYear', 'PubDate', 'ATM', 'BRCA1', 'BRCA2', 'PALB2', 'CDK12', 'CHEK2', 'PPP2R2A', 'RAD54L', 'BRIP1', 'BARD1', 'CHEK1', 'FANCL', 'RAD51B', 'RAD51C', 'RAD51D', 'Sum', 'LLama31-70b', 'LLama33-70b', 'DeepSeek_V3', 'DeepSeek-R1-Distill-Llama-70B', 'Sum.1']


,PaperId,PaperTitle,Abstract,Lang,PubYear,PubDate,ATM,BRCA1,BRCA2,PALB2,CDK12,CHEK2,PPP2R2A,RAD54L,BRIP1,BARD1,CHEK1,FANCL,RAD51B,RAD51C,RAD51D,Sum,Human,LLama31-70b,LLama33-70b,DeepSeek_V3,DeepSeek-R1-Distill-Llama-70B,Sum.1
0,4403747873,Germline DNA damage repair variants and prognosis of patients with high-risk or metastatic prostate cancer,Abstract Purpose: Deleterious germline variants in certain DNA repair genes are risk factors for developing aggressive prostate cancer. The objective was to quantify their prognostic impact after ...,en,2024,45590,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,4404066987,The Current Status of Comprehensive Genomic Profiling in the Management of Metastatic Castration-Resistant Prostate Cancer: A Study from a Cooperative Hospital for Cancer Genomic Medicine in Japan,"Background: Several effective treatment modalities against metastatic castration-resistant prostate cancer (mCRPC) are available; however, an unmet clinical need persists for mCRPC treatment becau...",en,2024,45590,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,4403382046,Skin metastasis of BRCA mutated prostate cancer: A case report and a brief review of literature,"Rationale: Metastatic castration-resistant prostate cancer has a poor prognosis especially when harboring DNA damage repair gene mutations, nevertheless, in the case of pathogenic BRCA gene mutati...",en,2024,45576,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
3,4403130090,"CDK12 loss drives prostate cancer progression, transcription-replication conflicts, and synthetic lethality with paralog CDK13","Biallelic loss of cyclin-dependent kinase 12 (CDK12) defines a metastatic castration-resistant prostate cancer (mCRPC) subtype. It remains unclear, however, whether CDK12 loss drives prostate canc...",en,2024,45566,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,4402986712,TMPRSS2:ERGGene Fusion Might Predict Resistance to PARP Inhibitors in Metastatic Castration-resistant Prostate Cancer,The emergence of novel DNA damage repair (DDR) pathways in molecular-target therapy drugs (MTTD) has shown promising outcomes in treating patients with metastatic castration-resistant prostate can...,en,2024,45565,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0


## 3. Choose tokenizer(s) to inspect

In [5]:
# Model/tokenizer mapping from your previous notebook.
# Add or remove tokenizers as needed.

model_fullnames = {
    "llama31-70b": "meta-llama/Meta-Llama-3.1-70B-Instruct",
    "llama33-70b": "meta-llama/Llama-3.3-70B-Instruct",
    "deepseek_v3": "deepseek-ai/DeepSeek-V3",
    "deepseek_r1": "deepseek-ai/DeepSeek-R1",
    "deepseek_r1_distill_llama_70b": "deepseek-ai/DeepSeek-R1-Distill-Llama-70B",

    # Optional biomedical tokenizers for comparison:
    # "biobert": "dmis-lab/biobert-base-cased-v1.1",
    # "pubmedbert": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext",
}

# Pick one main tokenizer for the main analysis.
TOKENIZER_KEY = "llama33-70b"

hf_model_name = model_fullnames[TOKENIZER_KEY]
print("Loading tokenizer:", TOKENIZER_KEY, "->", hf_model_name)

tokenizer = AutoTokenizer.from_pretrained(
    hf_model_name,
    use_fast=True,
    trust_remote_code=True
)

print("Tokenizer loaded.")

Loading tokenizer: llama33-70b -> meta-llama/Llama-3.3-70B-Instruct
Tokenizer loaded.


## 4. Core tokenizer diagnostic functions

In [6]:
def inspect_tokenization(text, tokenizer):
    """
    Return a token-level view of a string.
    This is the main function for seeing exactly how a variant is split.
    """
    text = str(text)

    enc = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = enc["input_ids"]
    offsets = enc["offset_mapping"]
    tokens = tokenizer.convert_ids_to_tokens(input_ids)

    rows = []
    for i, (tok, tok_id, offset) in enumerate(zip(tokens, input_ids, offsets)):
        start, end = offset
        rows.append({
            "token_index": i,
            "token": tok,
            "token_id": tok_id,
            "char_start": start,
            "char_end": end,
            "text_piece": text[start:end],
        })

    return pd.DataFrame(rows)


def tokenization_features(variant, tokenizer):
    """
    Summarize how tokenizer-hostile a variant string is.
    Runs tokenization both raw and with a leading space, because many BPE /
    SentencePiece tokenizers behave differently at word boundaries.
    """
    variant = str(variant).strip()

    result = {
        "variant": variant,
        "n_chars": len(variant),
        "has_punctuation": bool(re.search(r"[.\->_/+:]", variant)),
        "has_digit": bool(re.search(r"\d", variant)),
        "has_mixed_case": bool(re.search(r"[a-z]", variant) and re.search(r"[A-Z]", variant)),
        "has_hgvs_like_prefix": bool(re.search(r"\b[cpgnmr]\.", variant, flags=re.IGNORECASE)),
        "has_reference_sequence": bool(re.search(r"\bN[MR]_\d+(?:\.\d+)?", variant)),
        "has_rs_id": bool(re.search(r"\brs\d+\b", variant, flags=re.IGNORECASE)),
    }

    for mode, text in {
        "raw": variant,
        "leading_space": " " + variant
    }.items():
        enc = tokenizer(
            text,
            add_special_tokens=False,
            return_offsets_mapping=True
        )

        tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
        pieces = [text[s:e] for s, e in enc["offset_mapping"]]

        n_tokens = len(tokens)
        n_chars = len(text)

        result[f"n_tokens_{mode}"] = n_tokens
        result[f"chars_per_token_{mode}"] = n_chars / n_tokens if n_tokens else np.nan

        if mode == "raw":
            result["tokens_raw"] = tokens
            result["pieces_raw"] = pieces

    result["fragmentation_ratio"] = result["n_tokens_raw"] / max(len(variant), 1)

    return result


def parse_variant_cell(x):
    """
    Try to extract variant strings from a cell.

    Handles:
    - empty / zero / no-variant cells
    - Python-list-like strings
    - semicolon/newline/pipe-separated lists
    - plain strings
    """
    if pd.isna(x):
        return []

    x = str(x).strip()

    no_variant_values = {
        "", "0", "none", "nan", "no variant", "no variants",
        "no genetic variant", "no genetic variants",
        "no genetic variant detected", "no genetic variant detected in this publication."
    }

    if x.lower() in no_variant_values:
        return []

    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, (list, tuple, set)):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass

    parts = re.split(r";|\n|\|", x)
    parts = [p.strip() for p in parts if p.strip()]

    return parts


def normalize_variant(v):
    """
    Lightweight string normalization for variant matching.
    This is intentionally conservative.
    """
    v = str(v).strip()
    v = re.sub(r"\s+", " ", v)
    v = v.replace("c. ", "c.")
    v = v.replace("p. ", "p.")
    return v.lower()

## 5. Inspect tokenization of representative variant examples

In [7]:
example_variants = [
    "BRCA1 c.68_69delAG",
    "c.5266dupC",
    "p.Val600Glu",
    "V600E",
    "BRAF V600E",
    "rs121913529",
    "185delAG",
    "5382insC",
    "EGFR exon 19 deletion",
    "c.1799T>A",
    "p.Gly12Asp",
    "KRAS G12D",
    "NM_004333.6:c.1799T>A",
    "BRCA2 6174delT",
    "IVS2+1G>A",
]

example_feature_df = pd.DataFrame([
    tokenization_features(v, tokenizer)
    for v in example_variants
]).sort_values("n_tokens_raw", ascending=False)

display(example_feature_df[
    [
        "variant", "n_chars", "n_tokens_raw", "chars_per_token_raw",
        "fragmentation_ratio", "has_punctuation", "has_hgvs_like_prefix",
        "has_reference_sequence", "tokens_raw"
    ]
])

,variant,n_chars,n_tokens_raw,chars_per_token_raw,fragmentation_ratio,has_punctuation,has_hgvs_like_prefix,has_reference_sequence,tokens_raw
12,NM_004333.6:c.1799T>A,21,12,1.750000,0.571429,True,True,True,"[NM, _, 004, 333, ., 6, :c, ., 179, 9, T, >A]"
0,BRCA1 c.68_69delAG,18,10,1.800000,0.555556,True,True,False,"[BR, CA, 1, Ġc, ., 68, _, 69, del, AG]"
13,BRCA2 6174delT,14,8,1.750000,0.571429,False,False,False,"[BR, CA, 2, Ġ, 617, 4, del, T]"
14,IVS2+1G>A,9,7,1.285714,0.777778,True,False,False,"[IV, S, 2, +, 1, G, >A]"
1,c.5266dupC,10,6,1.666667,0.600000,True,True,False,"[c, ., 526, 6, dup, C]"
4,BRAF V600E,10,6,1.666667,0.600000,False,False,False,"[B, RA, F, ĠV, 600, E]"
8,EGFR exon 19 deletion,21,6,3.500000,0.285714,False,False,False,"[EG, FR, Ġexon, Ġ, 19, Ġdeletion]"
9,c.1799T>A,9,6,1.500000,0.666667,True,True,False,"[c, ., 179, 9, T, >A]"
10,p.Gly12Asp,10,6,1.666667,0.600000,True,True,False,"[p, .G, ly, 12, As, p]"
2,p.Val600Glu,11,5,2.200000,0.454545,True,True,False,"[p, .Val, 600, G, lu]"


In [ ]:
# Change this string to inspect any variant manually.
VARIANT_TO_INSPECT = "NM_004333.6:c.1799T>A"

display(inspect_tokenization(VARIANT_TO_INSPECT, tokenizer))

## 6. Evaluate normal binary performance first

In [11]:
def binary_label(x):
    """
    Convert evaluation cells to binary labels.
    Matches the logic of your original notebook:
    x == '0' -> 0, anything else -> 1
    """
    if pd.isna(x):
        return 0
    return 0 if str(x).strip() == "0" else 1


llm_df_bn = llm_df.copy()

model_columns = ['LLama31-70b', 'LLama33-70b', 'DeepSeek_V3', 'DeepSeek-R1-Distill-Llama-70B']
binary_columns = ["Human"] + model_columns
for col in binary_columns:
    llm_df_bn[col] = llm_df_bn[col].map(binary_label)

performance_rows = []

for model_col in model_columns:
    y_true = llm_df_bn["Human"]
    y_pred = llm_df_bn[model_col]

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    precision = tp / (tp + fp) if (tp + fp) else np.nan
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    accuracy = (tp + tn) / (tp + fp + tn + fn) if (tp + fp + tn + fn) else np.nan
    f1 = f1_score(y_true, y_pred, zero_division=0)

    performance_rows.append({
        "model": model_col,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "accuracy": accuracy,
        "f1": f1,
    })

performance_df = pd.DataFrame(performance_rows).sort_values("recall", ascending=True)

display(performance_df)

,model,TP,FP,FN,TN,precision,recall,specificity,accuracy,f1
1,LLama33-70b,21,0,2,774,1.000000,0.913043,1.000000,0.997491,0.954545
2,DeepSeek_V3,21,0,2,774,1.000000,0.913043,1.000000,0.997491,0.954545
3,DeepSeek-R1-Distill-Llama-70B,21,5,2,769,0.807692,0.913043,0.993540,0.991217,0.857143
0,LLama31-70b,23,6,0,768,0.793103,1.000000,0.992248,0.992472,0.884615


In [12]:
# Choose the model column for detailed false-negative analysis.
# Default: model with the lowest recall.

if len(performance_df) > 0:
    MODEL_FOR_ERROR_ANALYSIS = performance_df.iloc[0]["model"]
else:
    MODEL_FOR_ERROR_ANALYSIS = model_columns[0]

print("Model selected for detailed error analysis:", MODEL_FOR_ERROR_ANALYSIS)

def classify_error_type(row, model_col):
    human = row["Human"]
    pred = row[model_col]

    if human == 1 and pred == 1:
        return "TP"
    if human == 1 and pred == 0:
        return "FN"
    if human == 0 and pred == 1:
        return "FP"
    return "TN"

error_df = llm_df_bn.copy()
error_df["error_type"] = error_df.apply(
    lambda r: classify_error_type(r, MODEL_FOR_ERROR_ANALYSIS),
    axis=1
)

display(error_df["error_type"].value_counts().rename("count").to_frame())

Model selected for detailed error analysis: LLama33-70b


,count
error_type,
TN,774
TP,21
FN,2


## 7. Variant-level analysis if the evaluation cells contain actual variant strings

This works best if `Human` and model columns contain variant names or lists of variant names.

If the file contains only binary labels, the notebook skips this exact variant-level section and uses candidate extraction from abstracts instead.

In [13]:
def looks_binary_only(series):
    values = set(str(x).strip() for x in series.dropna().unique())
    return values.issubset({"0", "1"})


human_looks_binary = looks_binary_only(llm_df["Human"])
model_looks_binary = looks_binary_only(llm_df[MODEL_FOR_ERROR_ANALYSIS])

print("Human column looks binary-only:", human_looks_binary)
print(f"{MODEL_FOR_ERROR_ANALYSIS} column looks binary-only:", model_looks_binary)

can_do_variant_level_eval = not (human_looks_binary and model_looks_binary)
print("Can attempt variant-level evaluation:", can_do_variant_level_eval)

Human column looks binary-only: False
LLama33-70b column looks binary-only: False
Can attempt variant-level evaluation: True


In [14]:
def variant_level_eval_for_model(df, model_col, gold_col="Human", id_col="PaperId"):
    rows = []

    for _, row in df.iterrows():
        paper_id = row[id_col] if id_col in df.columns else None

        gold_raw = parse_variant_cell(row[gold_col])
        pred_raw = parse_variant_cell(row[model_col])

        gold = {normalize_variant(v): v for v in gold_raw}
        pred = {normalize_variant(v): v for v in pred_raw}

        for v_norm, v_original in gold.items():
            rows.append({
                "PaperId": paper_id,
                "variant_norm": v_norm,
                "variant_original": v_original,
                "status": "TP" if v_norm in pred else "FN",
                "model": model_col,
            })

        for v_norm, v_original in pred.items():
            if v_norm not in gold:
                rows.append({
                    "PaperId": paper_id,
                    "variant_norm": v_norm,
                    "variant_original": v_original,
                    "status": "FP",
                    "model": model_col,
                })

    return pd.DataFrame(rows)


if can_do_variant_level_eval:
    variant_eval_df = variant_level_eval_for_model(llm_df, MODEL_FOR_ERROR_ANALYSIS)

    if len(variant_eval_df) > 0:
        unique_variants = variant_eval_df["variant_original"].dropna().unique()

        variant_feature_df = pd.DataFrame([
            tokenization_features(v, tokenizer)
            for v in unique_variants
        ])

        variant_eval_tok_df = variant_eval_df.merge(
            variant_feature_df,
            left_on="variant_original",
            right_on="variant",
            how="left"
        )

        display(variant_eval_tok_df.head())
        display(
            variant_eval_tok_df
            .groupby("status")[["n_tokens_raw", "chars_per_token_raw", "fragmentation_ratio"]]
            .describe()
        )
    else:
        print("No variant-level rows were created.")
else:
    print("Skipping variant-level exact matching because the evaluation columns appear binary-only.")

<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal
<unknown>:1: SyntaxWarning: invalid decimal literal


,PaperId,variant_norm,variant_original,status,model,variant,n_chars,has_punctuation,has_digit,has_mixed_case,has_hgvs_like_prefix,has_reference_sequence,has_rs_id,n_tokens_raw,chars_per_token_raw,tokens_raw,pieces_raw,n_tokens_leading_space,chars_per_token_leading_space,fragmentation_ratio
0,4391302667,"ar t878a, ar l702h, ar h875y, ar f877l","AR T878A, AR L702H, AR H875Y, AR F877L",TP,LLama33-70b,"AR T878A, AR L702H, AR H875Y, AR F877L",39,False,True,False,False,False,False,20,1.950000,"[AR, ĠT, 878, A, ,, ĠAR, ĠL, 702, H, ,, ĠAR, ĠH, 875, Y, ,, Ġ, ĠAR, ĠF, 877, L]","[AR, T, 878, A, ,, AR, L, 702, H, ,, AR, H, 875, Y, ,, , AR, F, 877, L]",20,2.000000,0.512821
1,4390430980,p.asp427thrfs*3,p.Asp427Thrfs*3,TP,LLama33-70b,p.Asp427Thrfs*3,15,True,True,True,True,False,False,8,1.875000,"[p, .A, sp, 427, Thr, fs, *, 3]","[p, .A, sp, 427, Thr, fs, *, 3]",8,2.000000,0.533333
2,4388760791,c.6341del,c.6341del,TP,LLama33-70b,c.6341del,9,True,True,False,True,False,False,5,1.800000,"[c, ., 634, 1, del]","[c, ., 634, 1, del]",5,2.000000,0.555556
3,4385802919,"w1692fs*3, r320w, c2385","W1692fs*3, R320W, C2385",TP,LLama33-70b,"W1692fs*3, R320W, C2385",23,False,True,True,False,False,False,14,1.642857,"[W, 169, 2, fs, *, 3, ,, ĠR, 320, W, ,, ĠC, 238, 5]","[W, 169, 2, fs, *, 3, ,, R, 320, W, ,, C, 238, 5]",14,1.714286,0.608696
4,4324136924,f133,F133,FN,LLama33-70b,F133,4,False,True,False,False,False,False,2,2.000000,"[F, 133]","[F, 133]",2,2.500000,0.500000


n_tokens_raw                                                     \
              count       mean        std  min   25%  50%    75%   max   
status                                                                   
FN              2.0   5.500000   4.949747  2.0  3.75  5.5   7.25   9.0   
TP             21.0  14.142857  11.464230  3.0  6.00  9.0  20.00  40.0   

       chars_per_token_raw                                               \
                     count      mean       std  min       25%       50%   
status                                                                    
FN                     2.0  2.277778  0.392837  2.0  2.138889  2.277778   
TP                    21.0  1.803675  0.213448  1.5  1.666667  1.750000   

                           fragmentation_ratio                                \
             75%       max               count      mean       std       min   
status                                                                         
FN      2.416667  2.555556                 2.0  0.445652  0.076859  0.391304   
TP      1.875000  2.466667                21.0  0.560941  0.058575  0.405405   

                                                
             25%       50%       75%       max  
status                                          
FN      0.418478  0.445652  0.472826  0.500000  
TP      0.533333  0.571429  0.600000  0.666667

In [19]:
variant_eval_tok_df[variant_eval_tok_df['status'] == 'FN']

,PaperId,variant_norm,variant_original,status,model,variant,n_chars,has_punctuation,has_digit,has_mixed_case,has_hgvs_like_prefix,has_reference_sequence,has_rs_id,n_tokens_raw,chars_per_token_raw,tokens_raw,pieces_raw,n_tokens_leading_space,chars_per_token_leading_space,fragmentation_ratio
4,4324136924,f133,F133,FN,LLama33-70b,F133,4,False,True,False,False,False,False,2,2.000000,"[F, 133]","[F, 133]",2,2.500000,0.500000
9,3211749669,spop f133 and spop f102,SPOP F133 and SPOP F102,FN,LLama33-70b,SPOP F133 and SPOP F102,23,False,True,True,False,False,False,9,2.555556,"[S, POP, ĠF, 133, Ġand, ĠS, POP, ĠF, 102]","[S, POP, F, 133, and, S, POP, F, 102]",9,2.666667,0.391304


## 8. Candidate variant extraction from titles and abstracts

Use this when your evaluation is binary-only.

This does not prove that a candidate string is the exact gold variant, but it helps answer:

> Do false-negative articles contain variant-like strings that are unusually fragmented by the tokenizer?

In [ ]:
VARIANT_REGEX = re.compile(
    r"""
    (?:
        \brs\d+\b
        |
        \b[cpgnmr]\.\d+(?:_\d+)?(?:[ACGT]>[ACGT]|del[A-Za-z0-9]*|dup[A-Za-z0-9]*|ins[A-Za-z0-9]*)\b
        |
        \bp\.[A-Z][a-z]{2}\d+[A-Z][a-z]{2}\b
        |
        \bp\.[A-Z]\d+[A-Z]\b
        |
        \bN[MR]_\d+(?:\.\d+)?:[cpgnmr]\.[A-Za-z0-9_>.+\-]+\b
        |
        \b[A-Z]{1,8}\s+[A-Z]\d+[A-Z]\b
        |
        \b[A-Z]\d+[A-Z]\b
        |
        \b\d+(?:del|ins|dup)[A-Za-z]*\b
        |
        \bIVS\d+[+\-]\d+[ACGT]>[ACGT]\b
    )
    """,
    flags=re.VERBOSE | re.IGNORECASE
)


def extract_candidate_variants(text):
    if pd.isna(text):
        return []

    text = str(text)
    candidates = [m.group(0).strip() for m in VARIANT_REGEX.finditer(text)]

    cleaned = []
    for c in candidates:
        if len(c) < 3:
            continue
        cleaned.append(c)

    return sorted(set(cleaned))


if "PaperId" not in error_df.columns:
    raise ValueError("PaperId is required for title/abstract candidate analysis.")

candidate_base_df = error_df[["PaperId", "error_type"]].merge(
    cancer_df,
    on="PaperId",
    how="left"
)

candidate_rows = []

for _, row in candidate_base_df.iterrows():
    text = f"{row.get('PaperTitle', '')} {row.get('Abstract', '')}"
    candidates = extract_candidate_variants(text)

    for cand in candidates:
        feats = tokenization_features(cand, tokenizer)
        feats.update({
            "PaperId": row["PaperId"],
            "error_type": row["error_type"],
            "candidate_variant": cand,
            "PaperTitle": row.get("PaperTitle", None),
        })
        candidate_rows.append(feats)

candidate_tok_df = pd.DataFrame(candidate_rows)

print(f"Candidate variant mentions found: {len(candidate_tok_df):,}")
display(candidate_tok_df.head())

In [ ]:
if len(candidate_tok_df) > 0:
    summary = (
        candidate_tok_df
        .groupby("error_type")
        .agg(
            n_candidate_mentions=("candidate_variant", "count"),
            n_articles=("PaperId", "nunique"),
            mean_tokens=("n_tokens_raw", "mean"),
            median_tokens=("n_tokens_raw", "median"),
            mean_fragmentation=("fragmentation_ratio", "mean"),
            pct_hgvs_like=("has_hgvs_like_prefix", "mean"),
            pct_reference_sequence=("has_reference_sequence", "mean"),
            pct_rs_id=("has_rs_id", "mean"),
        )
        .sort_index()
    )

    display(summary)

    print("Most fragmented candidate variants in false-negative articles:")
    display(
        candidate_tok_df[candidate_tok_df["error_type"].eq("FN")]
        .sort_values(["n_tokens_raw", "fragmentation_ratio"], ascending=False)
        [
            [
                "PaperId", "candidate_variant", "n_tokens_raw", "chars_per_token_raw",
                "fragmentation_ratio", "tokens_raw", "PaperTitle"
            ]
        ]
        .head(50)
    )
else:
    print("No candidate variants found by the regex.")

## 9. Visualize tokenization complexity by error type

In [ ]:
if len(candidate_tok_df) > 0:
    plot_df = candidate_tok_df[candidate_tok_df["error_type"].isin(["TP", "FN", "FP", "TN"])].copy()

    grouped = plot_df.groupby("error_type")["n_tokens_raw"].mean().reindex(["TP", "FN", "FP", "TN"])

    plt.figure(figsize=(7, 4))
    plt.bar(grouped.index.astype(str), grouped.values)
    plt.title(f"Mean token count per candidate variant: {TOKENIZER_KEY}")
    plt.xlabel("Article-level error type")
    plt.ylabel("Mean number of tokens")
    plt.show()

    grouped_frag = plot_df.groupby("error_type")["fragmentation_ratio"].mean().reindex(["TP", "FN", "FP", "TN"])

    plt.figure(figsize=(7, 4))
    plt.bar(grouped_frag.index.astype(str), grouped_frag.values)
    plt.title(f"Mean fragmentation ratio per candidate variant: {TOKENIZER_KEY}")
    plt.xlabel("Article-level error type")
    plt.ylabel("Mean tokens per character")
    plt.show()
else:
    print("No candidate-tokenization data to plot.")

## 10. Compare tokenizers on the same variants

In [ ]:
# Select tokenizers to compare.
# Keep this list short if downloading tokenizers is slow in your environment.

TOKENIZERS_TO_COMPARE = [
    "llama31-70b",
    "llama33-70b",
    "deepseek_r1_distill_llama_70b",
    # "biobert",
    # "pubmedbert",
]

# Use the most fragmented FN candidates if available; otherwise use the examples.
if len(candidate_tok_df) > 0 and (candidate_tok_df["error_type"] == "FN").any():
    variants_to_compare = (
        candidate_tok_df[candidate_tok_df["error_type"].eq("FN")]
        .sort_values(["n_tokens_raw", "fragmentation_ratio"], ascending=False)
        ["candidate_variant"]
        .drop_duplicates()
        .head(25)
        .tolist()
    )
else:
    variants_to_compare = example_variants

comparison_rows = []

for tok_key in TOKENIZERS_TO_COMPARE:
    tok_name = model_fullnames[tok_key]
    print("Loading:", tok_key, "->", tok_name)

    tok = AutoTokenizer.from_pretrained(
        tok_name,
        use_fast=True,
        trust_remote_code=True
    )

    for variant in variants_to_compare:
        feats = tokenization_features(variant, tok)
        comparison_rows.append({
            "tokenizer": tok_key,
            "variant": variant,
            "n_tokens_raw": feats["n_tokens_raw"],
            "chars_per_token_raw": feats["chars_per_token_raw"],
            "fragmentation_ratio": feats["fragmentation_ratio"],
            "tokens_raw": feats["tokens_raw"],
        })

tokenizer_comparison_df = pd.DataFrame(comparison_rows)

display(
    tokenizer_comparison_df
    .sort_values(["variant", "n_tokens_raw"])
)

In [ ]:
if len(tokenizer_comparison_df) > 0:
    comparison_summary = (
        tokenizer_comparison_df
        .groupby("tokenizer")
        .agg(
            mean_tokens=("n_tokens_raw", "mean"),
            median_tokens=("n_tokens_raw", "median"),
            mean_fragmentation=("fragmentation_ratio", "mean"),
        )
        .sort_values("mean_tokens")
    )

    display(comparison_summary)

    plt.figure(figsize=(8, 4))
    plt.bar(comparison_summary.index.astype(str), comparison_summary["mean_tokens"].values)
    plt.title("Mean token count across selected variants")
    plt.xlabel("Tokenizer")
    plt.ylabel("Mean number of tokens")
    plt.xticks(rotation=45, ha="right")
    plt.show()
else:
    print("No tokenizer comparison data available.")

## 11. Export results

In [ ]:
# Save outputs for inspection.
output_path.mkdir(parents=True, exist_ok=True)

performance_out = output_path / "tokenizer_analysis_binary_performance.csv"
candidate_out = output_path / "tokenizer_analysis_candidate_variants.csv"
comparison_out = output_path / "tokenizer_analysis_tokenizer_comparison.csv"

performance_df.to_csv(performance_out, index=False)

if "candidate_tok_df" in globals() and len(candidate_tok_df) > 0:
    candidate_tok_df.to_csv(candidate_out, index=False)

if "tokenizer_comparison_df" in globals() and len(tokenizer_comparison_df) > 0:
    tokenizer_comparison_df.to_csv(comparison_out, index=False)

print("Saved:")
print(performance_out)
if "candidate_tok_df" in globals() and len(candidate_tok_df) > 0:
    print(candidate_out)
if "tokenizer_comparison_df" in globals() and len(tokenizer_comparison_df) > 0:
    print(comparison_out)

## 12. How to interpret the result

Evidence that tokenization may be hurting recall:

- FN articles have candidate variants with more tokens than TP articles.
- FN candidates have a higher fragmentation ratio.
- FN candidates are enriched for HGVS-like strings, reference-sequence strings, punctuation-heavy variants, or complex indels.
- The same variants are much less fragmented in another tokenizer.

Evidence against tokenizer being the main issue:

- TP and FN groups have similar token counts and fragmentation ratios.
- Misses are mostly simple variants like `V600E`, `G12D`, or `rs...`.
- Errors correlate more with abstract ambiguity, prompt parsing, output formatting, or evaluation strictness.